# 🏥 Medical LLM Assistant — Tech Challenge Fase 3

**Autor:** Felipe Huszar  
**Data:** Março 2026  

Assistente médico com LLM (Mistral fine-tuned via LoRA), orquestrado por **LangGraph** e memória via **ChromaDB**.

## 📋 Sumário
1. [Clone do Repositório](#clone)
2. [Instalação de Dependências](#deps)
3. [Configuração](#config)
4. [Execução de Testes](#tests)
5. [Demonstração do Pipeline](#demo)
6. [Interface Gradio](#ui)

## 🔧 Modos de Execução
- **MockLLM** (`USE_MOCK_LLM='true'`): Rápido, sem GPU, respostas determinísticas
- **Modelo Real** (`USE_MOCK_LLM='false'`): Requer GPU T4+ e LoRA no Drive

## 1️⃣ Clone do Repositório

In [ ]:
# Clone ou atualiza o repositório
# Substitua pela URL do seu repo se for privado
!git clone https://github.com/felipe-huszar/tech-challenge-fase3.git medical-llm-assistant 2>/dev/null || \
  (cd medical-llm-assistant && git pull)

%cd medical-llm-assistant
!ls -la

## 2️⃣ Instalação de Dependências

In [ ]:
# Instalar dependências
!pip install -q -r requirements.txt

# Verificar instalações principais
import importlib
packages = ['langgraph', 'chromadb', 'gradio', 'transformers', 'torch']
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'unknown')
        print(f"✅ {pkg}: {version}")
    except ImportError:
        print(f"❌ {pkg}: NÃO instalado")

## 3️⃣ Configuração

In [ ]:
# Montar Google Drive (necessário apenas para modelo real)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Drive montado com sucesso.')
except ModuleNotFoundError:
    print('ℹ️ Não está no Colab — Drive não montado.')

In [ ]:
# Configuração de ambiente
import os
import sys

# ⬇️ Mude para 'false' para usar modelo real (requer GPU + LoRA no Drive)
os.environ['USE_MOCK_LLM'] = 'true'

# Caminho do adapter LoRA no Drive (ignorado se mock=true)
os.environ['MODEL_PATH'] = '/content/drive/MyDrive/medical_llm_lora'

# ID do modelo base HuggingFace (ignorado se mock=true)
os.environ['BASE_MODEL_ID'] = 'mistralai/Mistral-7B-Instruct-v0.1'

# Diretório ChromaDB
os.environ['CHROMA_DB_PATH'] = '/content/chroma_db'

# Adicionar raiz do projeto ao path
sys.path.insert(0, '/content/medical-llm-assistant')

print('='*60)
print('🔧 CONFIGURAÇÃO ATUAL')
print('='*60)
print(f"USE_MOCK_LLM  = {os.environ['USE_MOCK_LLM']}")
print(f"MODEL_PATH    = {os.environ['MODEL_PATH']}")
print(f"BASE_MODEL_ID = {os.environ['BASE_MODEL_ID']}")
print(f"CHROMA_DB_PATH= {os.environ['CHROMA_DB_PATH']}")
print('='*60)

## 4️⃣ Execução de Testes

In [ ]:
# Executar suite de testes completa
print('🧪 Executando testes...')
print('='*60)

# Testes Unitários
print('\n📦 Testes Unitários (Safety Gate)')
!python -m pytest tests/unit -v --tb=short

# Testes de Integração
print('\n📦 Testes de Integração (Pipeline)')
!python -m pytest tests/integration -v --tb=short

# Testes E2E
print('\n📦 Testes E2E (Jornadas Completas)')
!python -m pytest tests/e2e -v --tb=short

print('\n' + '='*60)
print('✅ Testes finalizados!')

## 5️⃣ Demonstração do Pipeline

In [ ]:
# Importar e configurar
from src.graph.pipeline import run_consultation
from src.rag.patient_rag import seed_from_file, get_patient, get_consultation_history

# Popular pacientes de exemplo
n = seed_from_file('data/patients_seed.json')
print(f'✅ Pacientes seed inseridos: {n}')

print('\n' + '='*60)
print('🩺 CONSULTA 1: Paciente com sintomas gastrointestinais')
print('='*60)

result1 = run_consultation(
    cpf='123.456.789-00',
    doctor_question='Paciente relata dores abdominais ao evacuar há 3 semanas, com sangue nas fezes ocasionalmente. Quais diagnósticos e exames considerar?',
)

print(f"\nStatus: {'✅ Aprovado' if result1['safety_passed'] else '⚠️ Escalado'}")
print(f"Escalar: {result1['needs_escalation']}")
print(f"Fontes: {result1['sources']}")
print('\n--- RESPOSTA ---\n')
print(result1['final_answer'])

In [ ]:
# Segunda consulta para o mesmo paciente (testar histórico)
print('\n' + '='*60)
print('🩺 CONSULTA 2: Retorno do mesmo paciente')
print('='*60)

result2 = run_consultation(
    cpf='123.456.789-00',
    doctor_question='Retorno: Paciente realizou colonoscopia que mostrou pólipos. Dor abdominal persistiu. Próximos passos?',
)

print(f"\nStatus: {'✅ Aprovado' if result2['safety_passed'] else '⚠️ Escalado'}")
print('\n--- RESPOSTA ---\n')
print(result2['final_answer'])

# Verificar histórico
history = get_consultation_history('123.456.789-00')
print(f"\n📋 Histórico do paciente: {len(history)} consulta(s)"
print('='*60)

In [ ]:
# Teste com sintomas cardiovasculares
print('\n' + '='*60)
print('🩺 CONSULTA 3: Sintomas cardiovasculares')
print('='*60)

result3 = run_consultation(
    cpf='987.654.321-00',
    doctor_question='Paciente de 65 anos com dor torácica em aperto, irradiação para braço esquerdo, dispneia ao esforço. ECG mostra supra ST.',
    patient_profile={'nome': 'Carlos Cardio', 'idade': 65, 'sexo': 'M', 'peso': 82},
)

print(f"\nStatus: {'✅ Aprovado' if result3['safety_passed'] else '⚠️ Escalado'}")
print('\n--- RESPOSTA ---\n')
print(result3['final_answer'])

In [ ]:
# Teste de segurança: tentativa de prescrição
print('\n' + '='*60)
print('🛡️ TESTE DE SEGURANÇA: Tentativa de prescrição')
print('='*60)

from unittest.mock import MagicMock
import json

# Simular LLM tentando prescrever
mock_llm = MagicMock()
mock_llm.invoke.return_value = json.dumps({
    'possible_diagnoses': ['Infecção bacteriana'],
    'recommended_exams': ['Hemocultura'],
    'reasoning': 'Prescrever amoxicilina 500mg 8/8h por 7 dias.',
    'sources': ['Protocolo'],
    'confidence': 0.85,
    'recommendation_type': 'prescription',  # PROIBIDO!
})

result_safety = run_consultation(
    cpf='SAFETY.TEST.001',
    doctor_question='Prescreva antibiótico para infecção.',
    patient_profile={'nome': 'Safety Test', 'idade': 40, 'sexo': 'M', 'peso': 75},
    llm=mock_llm,
)

print(f"\nStatus: {'✅ Aprovado' if result_safety['safety_passed'] else '⚠️ Escalado'}")
print(f"Escalar: {result_safety['needs_escalation']}")
print('\n--- RESPOSTA ---\n')
print(result_safety['final_answer'])

## 6️⃣ Interface Gradio (UI Completa)

In [ ]:
# Lançar interface Gradio
from app import demo

print('🚀 Iniciando interface Gradio...')
print('Clique no link abaixo para acessar:')
print('='*60)

demo.launch(share=True, debug=True)

## 📊 Extras: Análise dos Testes E2E

In [ ]:
# Análise da cobertura de testes
test_categories = {
    'Unit Tests (Safety Gate)': 13,
    'Integration Tests (Pipeline)': 12,
    'E2E - Core Journeys': 10,
    'E2E - Pipeline': 11,
    'E2E - Extended': 32,
}

total = sum(test_categories.values())

print('='*60)
print('📊 COBERTURA DE TESTES')
print('='*60)
for category, count in test_categories.items():
    print(f"{category:.<40} {count:>3} testes")
print('-'*60)
print(f"{'TOTAL':.<40} {total:>3} testes")
print('='*60)

In [ ]:
# Visualização da arquitetura
print('='*60)
print('🏗️ ARQUITETURA DO SISTEMA')
print('='*60)
print('''
[CPF input]
     ↓
[Nó 1] check_patient    → busca no ChromaDB por CPF
     ↓
[Nó 2] retrieve_history → histórico de consultas
     ↓
[Nó 3] build_prompt     → perfil + histórico + pergunta
     ↓
[Nó 4] llm_reasoning    → MockLLM ou Mistral LoRA
     ↓
[Nó 5] safety_gate      → validação de segurança
     ↓
[Nó 6] save_and_format  → salva no ChromaDB + formata
     ↓
[END]  final_answer     → resposta ao médico

     ⚠️ Se safety_gate falhar → escalation
''')
print('='*60)